# Модуль 2 · Лекція 2 — Функції та область видимості (LEGB)

**Передумова:** [Лекція 1 — цикли, ітератори, генератори](lektsiya-1-umovni-operatory-tsykly.ipynb)

## Що ви знатимете після лекції
- функції: параметри позиційні/іменовані/за замовчуванням, `*args`, `**kwargs`;
- ⚠️ пастка **змінюваного аргументу за замовчуванням** (`def f(x, lst=[])`);
- функції як **об'єкти першого класу**, `lambda`, `map`/`filter`;
- **область видимості LEGB**, `global`, `nonlocal`, `UnboundLocalError`;
- **замикання (closures)** і чому вони важливі;
- **декоратори** — від нуля до `functools.wraps`;
- **менеджери контексту** (`with`, `__enter__`/`__exit__`, `contextlib`);
- 🎁 бонус: `asyncio` та багатопоточність/багатопроцесність (concurrency) — оглядово.

Наприкінці — **⚡ каверзні питання** по всіх темах лекції.

## 1. Функції: параметри та аргументи

- **Параметр** — імʼя у визначенні функції; **аргумент** — конкретне значення при виклику.
- Аргументи бувають **позиційні** (за порядком) та **іменовані/keyword** (за імʼям).
- Параметри можуть мати **значення за замовчуванням**.

In [ ]:
def create_user(name, age, city="Львів", is_active=True):
    return f"{name}, {age} р., {city}, active={is_active}"

print(create_user("Іван", 30))                       # позиційні + замовчування
print(create_user("Марія", 25, city="Київ"))         # іменований аргумент
print(create_user(age=40, name="Олег"))              # порядок неважливий для іменованих

## 2. `*args` та `**kwargs` ⭐

Дозволяють функції приймати **довільну** кількість аргументів.

- **`*args`** збирає зайві **позиційні** аргументи в **кортеж**.
- **`**kwargs`** збирає зайві **іменовані** аргументи в **словник**.

Імена `args`/`kwargs` — лише конвенція; значення мають символи `*` і `**`.

In [ ]:
def demo(*args, **kwargs):
    print("args   (tuple):", args)
    print("kwargs (dict): ", kwargs)

demo(1, 2, 3, name="Аня", age=25)

In [ ]:
# Практично: сума будь-якої кількості чисел
def total(*numbers):
    return sum(numbers)

print(total(1, 2, 3, 4, 5))

# Розпакування (unpacking) при ВИКЛИКУ — дзеркало *args/**kwargs
def point(x, y, z):
    return (x, y, z)

coords = [1, 2, 3]
print(point(*coords))            # розпакувати список у позиційні

params = {"x": 10, "y": 20, "z": 30}
print(point(**params))           # розпакувати dict у іменовані

## 3. ⚠️ Пастка: змінюваний аргумент за замовчуванням

Одна з найвідоміших пасток Python. Значення за замовчуванням **обчислюється ОДИН раз** — у момент визначення функції, а не при кожному виклику. Якщо це змінюваний обʼєкт (`list`, `dict`), він **зберігається між викликами**.

In [ ]:
# ПОГАНО:
def add_item(item, basket=[]):     # basket створюється ОДИН раз!
    basket.append(item)
    return basket

print(add_item("яблуко"))          # ['яблуко']
print(add_item("банан"))           # ['яблуко', 'банан'] — сюрприз! той самий список

In [ ]:
# ДОБРЕ: як default використовуємо None, а список створюємо всередині
def add_item(item, basket=None):
    if basket is None:
        basket = []                # новий список при КОЖНОМУ виклику
    basket.append(item)
    return basket

print(add_item("яблуко"))          # ['яблуко']
print(add_item("банан"))           # ['банан'] — правильно

## 4. Функції — об'єкти першого класу

Функцію можна: присвоїти змінній, передати в іншу функцію як аргумент, повернути з функції, покласти в список/словник. Це основа декораторів і функційного стилю.

In [ ]:
def shout(text):
    return text.upper() + "!"

# функція у змінній
f = shout
print(f("привіт"))

# функція як аргумент (higher-order function)
def apply_twice(func, value):
    return func(func(value))

print(apply_twice(lambda x: x + 3, 10))   # 16

# lambda — анонімна функція одного виразу
add = lambda a, b: a + b
print(add(2, 3))

# map / filter (повертають ліниві ітератори)
nums = [1, 2, 3, 4, 5]
print(list(map(lambda x: x ** 2, nums)))          # [1,4,9,16,25]
print(list(filter(lambda x: x % 2 == 0, nums)))   # [2, 4]

## 5. Область видимості: правило LEGB ⭐

Коли Python бачить імʼя, він шукає його в чотирьох просторах саме в такому порядку:

1. **L — Local**: усередині поточної функції.
2. **E — Enclosing**: у зовнішній (охоплюючій) функції, якщо є вкладеність.
3. **G — Global**: на рівні модуля.
4. **B — Built-in**: вбудовані імена (`len`, `print`, `range`...).

Перше знайдене — виграє.

In [ ]:
x = "global"

def outer():
    x = "enclosing"
    def inner():
        x = "local"
        print("inner бачить:", x)   # local
    inner()
    print("outer бачить:", x)       # enclosing

outer()
print("модуль бачить:", x)          # global

### 5.1. `global` та `nonlocal`

За замовчуванням присвоєння `x = ...` **створює локальну** змінну. Щоб змінювати зовнішню:
- **`global x`** — працювати зі змінною рівня модуля;
- **`nonlocal x`** — працювати зі змінною охоплюючої функції.

In [ ]:
counter = 0

def increment():
    global counter        # без цього рядка буде UnboundLocalError
    counter += 1

increment()
increment()
print("counter:", counter)   # 2

def make_counter():
    count = 0
    def step():
        nonlocal count       # змінюємо змінну ОХОПЛЮЮЧОЇ функції
        count += 1
        return count
    return step

c1 = make_counter()
print(c1(), c1(), c1())      # 1 2 3

In [ ]:
# ⚠️ Класична пастка: UnboundLocalError
value = 10

def broken():
    # print(value)  # спробуй розкоментувати
    value = value + 1   # оскільки далі є присвоєння, value вважається ЛОКАЛЬНОЮ
    return value        # ...але праворуч вона ще не існує -> помилка

try:
    broken()
except UnboundLocalError as e:
    print("UnboundLocalError:", e)

## 6. Замикання (closures) ⭐

**Замикання** — це вкладена функція, яка «пам'ятає» змінні з охоплюючої області навіть після того, як зовнішня функція завершила роботу. Саме на замиканнях побудовані декоратори.

In [ ]:
def multiplier(factor):
    def multiply(number):
        return number * factor    # factor "замкнено" із зовнішньої функції
    return multiply

double = multiplier(2)
triple = multiplier(3)
print(double(10))   # 20
print(triple(10))   # 30

# Доказ, що замикання зберігає змінну:
print(double.__closure__[0].cell_contents)   # 2

In [ ]:
# ⚠️ Пастка пізнього звʼязування (late binding) у циклі
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])          # [2, 2, 2] — усі бачать останнє i!

# Виправлення: зафіксувати значення через аргумент за замовчуванням
funcs = [lambda i=i: i for i in range(3)]
print([f() for f in funcs])          # [0, 1, 2]

## 7. Декоратори ⭐

**Декоратор** — функція, що приймає іншу функцію та повертає нову функцію з доданою поведінкою (логування, кешування, перевірка доступу). Синтаксис `@decorator` — це просто `func = decorator(func)`.

Будуємо крок за кроком.

In [ ]:
# Крок 1: декоратор вручну
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> викликаю {func.__name__}")
        result = func(*args, **kwargs)
        print(f"<- {func.__name__} повернула {result}")
        return result
    return wrapper

def add(a, b):
    return a + b

add = with_logging(add)   # "обгортаємо" вручну
add(2, 3)

In [ ]:
# Крок 2: те саме через синтаксис @ — читабельніше
def with_logging(func):
    def wrapper(*args, **kwargs):
        print(f"-> {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- {result}")
        return result
    return wrapper

@with_logging          # еквівалент multiply = with_logging(multiply)
def multiply(a, b):
    return a * b

multiply(4, 5)

In [ ]:
# Крок 3: чому потрібен functools.wraps
import functools

def bad_decorator(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@bad_decorator
def greet():
    """Вітає користувача."""
    return "Привіт"

print("Без wraps:", greet.__name__, "|", greet.__doc__)   # 'wrapper' — метадані загубились!

def good_decorator(func):
    @functools.wraps(func)         # зберігає __name__, __doc__ оригіналу
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@good_decorator
def greet2():
    """Вітає користувача."""
    return "Привіт"

print("З wraps:  ", greet2.__name__, "|", greet2.__doc__)

In [ ]:
# Практичний декоратор: кеш (мемоізація) для повільної функції
import functools

@functools.lru_cache(maxsize=None)   # готовий декоратор зі стандартної бібліотеки
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(50))                        # миттєво, попри рекурсію
print(fib.cache_info())

## 8. Менеджери контексту (`with`) ⭐

`with` гарантує, що ресурс буде **коректно звільнено** (файл закрито, з'єднання розірвано, блокування знято) — навіть якщо станеться помилка.

Об'єкт-менеджер реалізує два методи:
- **`__enter__`** — виконується на вході в блок (повертає ресурс);
- **`__exit__`** — виконується на виході (навіть при винятку) — тут прибирання.

In [ ]:
import os

# Найпоширеніший приклад: файл сам закриється
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("Привіт із with!")
# тут файл вже гарантовано закритий, навіть якби сталася помилка всередині

with open("demo.txt", encoding="utf-8") as f:
    print(f.read())

os.remove("demo.txt")   # прибираємо за собою

In [ ]:
# Власний менеджер контексту через клас
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        import time
        self.elapsed = time.perf_counter() - self.start
        print(f"Зайняло {self.elapsed:.4f} c")
        return False    # False -> виняток (якщо був) не пригнічується

with Timer():
    total = sum(range(1_000_000))
print("сума порахована:", total)

In [ ]:
# Простіший спосіб — через contextlib
from contextlib import contextmanager

@contextmanager
def tag(name):
    print(f"<{name}>")
    yield                 # усе до yield — це __enter__, усе після — __exit__
    print(f"</{name}>")

with tag("b"):
    print("жирний текст")

## 9. 🎁 Бонус: асинхронність та багатозадачність (оглядово)

Ці теми — поглиблені; тут короткий, але правильний огляд для загального розуміння.

### 9.1. Concurrency vs Parallelism
- **Concurrency (конкурентність)** — задачі *чергуються* й прогресують «майже одночасно» (перемикання). Добре для **I/O-bound** задач (мережа, диск — там багато очікування).
- **Parallelism (паралелізм)** — задачі *справді* виконуються одночасно на різних ядрах CPU. Потрібно для **CPU-bound** задач (важкі обчислення).

### 9.2. Три підходи в Python і при чому тут GIL
| Підхід | Модуль | Для чого | GIL |
|---|---|---|---|
| Багатопоточність | `threading` | I/O-bound | ⚠️ GIL заважає паралелізму CPU |
| Багатопроцесність | `multiprocessing` | CPU-bound | обходить GIL (окремі процеси) |
| Асинхронність | `asyncio` | I/O-bound, тисячі з'єднань | один потік, кооперативно |

**GIL (Global Interpreter Lock)** — у CPython лише один потік одночасно виконує байткод. Тому `threading` **не** прискорює обчислення на CPU, але **допомагає** при очікуванні I/O (поки один потік чекає мережу, інший працює).

In [ ]:
# asyncio: async/await. Корутина "віддає керування" на await,
# і поки вона "спить", встигають інші.
import asyncio

async def fetch(name, delay):
    print(f"  {name}: старт")
    await asyncio.sleep(delay)     # імітація I/O; НЕ блокує весь потік
    print(f"  {name}: готово за {delay}с")
    return name

async def main():
    # усі три "чекання" йдуть паралельно у часі -> ~1с, а не 1+0.5+0.7
    results = await asyncio.gather(
        fetch("A", 1.0),
        fetch("B", 0.5),
        fetch("C", 0.7),
    )
    print("результати:", results)

await main()   # у Jupyter можна await напряму; у звичайному .py -> asyncio.run(main())

In [ ]:
# CPU-bound: threading НЕ прискорить (GIL), multiprocessing — так.
# Тут просто демонструємо API потоків для I/O-подібної задачі.
import threading, time

def worker(n):
    time.sleep(0.2)          # імітація I/O-очікування
    print(f"  потік {n} завершив")

threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
for t in threads:
    t.start()
for t in threads:
    t.join()                 # чекаємо завершення всіх
print("усі потоки завершились")

## ⚡ Каверзні питання

In [ ]:
# Q1. Що надрукується і чому?
def append_to(item, target=[]):
    target.append(item)
    return target

print(append_to(1))   # ?
print(append_to(2))   # ?

**Q1 (відповідь).** `[1]`, потім `[1, 2]`. Список за замовчуванням створюється **один раз** і зберігається між викликами. Правильно — `target=None` + створення всередині.

**Q2. Різниця між `*args` і `**kwargs`?**
`*args` збирає зайві **позиційні** аргументи в **кортеж**; `**kwargs` — зайві **іменовані** у **словник**. При виклику `*`/`**` навпаки **розпаковують** послідовність/словник в аргументи.

**Q3. Що таке декоратор і навіщо `functools.wraps`?**
Декоратор — функція, що обгортає іншу, додаючи поведінку без зміни її коду. `functools.wraps` копіює метадані (`__name__`, `__doc__`) оригіналу на обгортку, інакше вони губляться.

**Q4. Поясніть LEGB. Що таке `nonlocal` vs `global`?**
Порядок пошуку імені: Local → Enclosing → Global → Built-in. `global` дозволяє **присвоювати** змінну модуля, `nonlocal` — змінну **охоплюючої функції**.

In [ ]:
# Q5. Чому [2, 2, 2], а не [0, 1, 2]?
funcs = [lambda: i for i in range(3)]
print([f() for f in funcs])

**Q5 (відповідь).** Пізнє зв'язування (late binding): усі лямбди посилаються на **ту саму** змінну `i`, а до моменту виклику вона вже дорівнює `2`. Фікс: `lambda i=i: i`.

**Q6. Навіщо `with` / менеджер контексту?**
Гарантує звільнення ресурсу (`__exit__`) навіть при винятку — файли, з'єднання, блокування. Замінює ручний `try/finally`.

**Q7. Чому `threading` не прискорює обчислення на CPU?**
Через **GIL**: у CPython одночасно байткод виконує лише один потік. Для CPU-bound беруть `multiprocessing` (окремі процеси, кожен зі своїм GIL), для I/O-bound добре працюють `threading` та `asyncio`.

## 🎯 Практичні вправи

1. Напишіть функцію `describe(**kwargs)`, яка друкує всі передані іменовані параметри у форматі `ключ = значення`.
2. Напишіть функцію `maximum(*args)`, яка повертає найбільший з переданих аргументів (без `max`).
3. Виправте «пастку» у функції `def collect(x, into=[])` так, щоб кожен виклик починав з порожнього списку.
4. Напишіть декоратор `@timeit`, який друкує час виконання обгорнутої функції (використайте `time.perf_counter`).
5. Напишіть декоратор `@repeat(n)` (з параметром!), який викликає функцію `n` разів.
6. Реалізуйте власний менеджер контексту `Suppress`, який ігнорує задані типи винятків (аналог `contextlib.suppress`).
7. За допомогою замикання напишіть `make_accumulator()`, що повертає функцію, яка додає передане число до внутрішньої суми й повертає її.
8. (⭐) Поясніть на власному прикладі коду різницю між `global` і `nonlocal`.
9. (⭐, async) Напишіть дві корутини, що «сплять» різний час, і запустіть їх разом через `asyncio.gather`, вимірявши загальний час.

In [ ]:
# Місце для ваших вправ


## 📚 Додаткові матеріали
- Функції (туторіал, укр.): https://docs.python.org/uk/3/tutorial/controlflow.html#defining-functions
- Область видимості та `nonlocal`/`global`: https://docs.python.org/3/reference/executionmodel.html
- `functools` (wraps, lru_cache): https://docs.python.org/3/library/functools.html
- `contextlib`: https://docs.python.org/3/library/contextlib.html
- Real Python — Decorators: https://realpython.com/primer-on-python-decorators/
- Real Python — Closures / scope: https://realpython.com/python-scope-legb-rule/
- `asyncio` (офіційно): https://docs.python.org/3/library/asyncio.html
- Real Python — Async IO: https://realpython.com/async-io-python/
- Real Python — Threading vs Multiprocessing / GIL: https://realpython.com/python-gil/